# M3 baselines vs human labels

Compare primary human labels (`human_annotation`) with Milestone 3 baselines on each row:

- **Phase 1:** `baseline_scores.m3.lexical.sequence_ratio`
- **Phase 2 (optional):** `m3.bertscore.f1` if you ran with `--bertscore`
- **Phase 3 (optional):** `m3.nli.label` and `m3.nli.scores` (e.g. entailment prob) if you ran with `--nli`

**Prerequisite:** run from the repository root (or adjust `ROOT` below) and use `pip install -e '.[dev]'` or `PYTHONPATH=.` so `rde_eval` imports resolve.

Rebuild `results/samples_with_m3.jsonl` after changing phases, for example:

```bash
python scripts/run_baselines.py --input data/samples.jsonl --output results/samples_with_m3.jsonl
python scripts/run_baselines.py --input data/samples.jsonl --output results/samples_with_m3.jsonl --bertscore
python scripts/run_baselines.py --input data/samples.jsonl --output results/samples_with_m3.jsonl --nli --nli-model facebook/roberta-large-mnli
```

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent

PATH = ROOT / "results" / "samples_with_m3.jsonl"
if not PATH.is_file():
    PATH = ROOT / "data" / "samples.jsonl"
    print("Using data/samples.jsonl — run run_baselines.py to create results/samples_with_m3.jsonl")

print("ROOT =", ROOT)
print("JSONL =", PATH)

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


rows = load_jsonl(PATH)
len(rows)

In [ ]:
from rde_eval.baselines import merge_milestone3_baseline

prepared: list[dict] = []
for r in rows:
    r = dict(r)
    if "baseline_scores" not in r or "m3" not in (r.get("baseline_scores") or {}):
        r["baseline_scores"] = merge_milestone3_baseline(
            r.get("baseline_scores"), source=str(r["source"]), output=str(r["output"])
        )
    prepared.append(r)

rows = prepared

## Mean lexical ratio by human primary label

Exploratory: higher overlap does not imply RDE "Preserved" — this is only a string-similarity proxy.

In [ ]:
by_label: dict[str, list[float]] = defaultdict(list)
for r in rows:
    label = r.get("human_annotation")
    if not label:
        continue
    ratio = r["baseline_scores"]["m3"]["lexical"]["sequence_ratio"]
    by_label[str(label)].append(ratio)

for label in sorted(by_label):
    vals = by_label[label]
    mean = sum(vals) / len(vals)
    print(f"{label:22}  mean_ratio={mean:.4f}  n={len(vals)}")

## Optional: BERTScore F1 and NLI label by `human_annotation`

If the JSONL was produced with `--bertscore` or `--nli`, the next cell prints simple aggregates. Otherwise it prints a short hint.

In [ ]:
from collections import Counter, defaultdict

by_h_f1: dict[str, list[float]] = defaultdict(list)
by_h_nli: dict[str, list[str]] = defaultdict(list)

for r in rows:
    ha = r.get("human_annotation")
    if not ha:
        continue
    ha = str(ha)
    m3 = (r.get("baseline_scores") or {}).get("m3") or {}
    bs = m3.get("bertscore") or {}
    if isinstance(bs, dict) and "f1" in bs:
        by_h_f1[ha].append(float(bs["f1"]))
    nl = m3.get("nli") or {}
    if isinstance(nl, dict) and nl.get("label"):
        by_h_nli[ha].append(str(nl["label"]))

if by_h_f1:
    print("BERTScore F1 (mean) by human_annotation")
    for lbl in sorted(by_h_f1):
        vals = by_h_f1[lbl]
        print(f"{lbl:22}  mean_f1={sum(vals) / len(vals):.4f}  n={len(vals)}")
else:
    print("No m3.bertscore in rows — rerun run_baselines with --bertscore if you want this block.")

print()
if by_h_nli:
    print("NLI predicted label counts by human_annotation (top 5 labels each)")
    for lbl in sorted(by_h_nli):
        ctr = Counter(by_h_nli[lbl])
        top = ", ".join(f"{k}:{v}" for k, v in ctr.most_common(5))
        print(f"{lbl:22}  {top}")
else:
    print("No m3.nli in rows — rerun run_baselines with --nli if you want this block.")

## Lexical vs BERTScore F1

If `--bertscore` was used alongside lexical baselines, we report **Pearson *r*** between `m3.lexical.sequence_ratio` and `m3.bertscore.f1`, and optionally a scatter plot when **matplotlib** is installed.

In [ ]:
import math


def pearson_r(xs: list[float], ys: list[float]) -> float:
    n = len(xs)
    if n != len(ys) or n < 2:
        return float("nan")
    mx = sum(xs) / n
    my = sum(ys) / n
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys, strict=True))
    dx = sum((x - mx) ** 2 for x in xs)
    dy = sum((y - my) ** 2 for y in ys)
    den = math.sqrt(dx * dy)
    return float("nan") if den == 0.0 else num / den


lex_vals: list[float] = []
bert_vals: list[float] = []
for r in rows:
    m3 = (r.get("baseline_scores") or {}).get("m3") or {}
    lr = (m3.get("lexical") or {}).get("sequence_ratio")
    bs = m3.get("bertscore") or {}
    bf = bs.get("f1")
    if lr is not None and bf is not None:
        lex_vals.append(float(lr))
        bert_vals.append(float(bf))

if len(lex_vals) >= 2:
    r_ab = pearson_r(lex_vals, bert_vals)
    msg = "Pearson r (lexical ratio, BERTScore F1) = {:.4f}  (n={})"
    print(msg.format(r_ab, len(lex_vals)))
else:
    print(
        "Need >=2 rows with both lexical + bertscore — rerun run_baselines with --bertscore."
    )

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is not None and len(lex_vals) >= 2:
    plt.figure(figsize=(4.0, 3.2))
    plt.scatter(lex_vals, bert_vals, alpha=0.75)
    plt.xlabel("lexical sequence_ratio")
    plt.ylabel("BERTScore F1")
    plt.tight_layout()
    plt.show()
elif len(lex_vals) >= 2:
    print("matplotlib not installed — scatter plot skipped")

## NLI softmax vs `human_annotation`

When `--nli` populated `m3.nli.scores`, we tabulate **mean softmax** for **`entailment`**, **`neutral`**, and **`contradiction`** (keys normalized to lowercase / underscores). Rows without `human_annotation` are skipped.

At the bottom we still report Pearson *r* between lexical `sequence_ratio` and **entailment** probability when enough paired rows exist.

In [ ]:
import math
from collections import defaultdict

ENT_KEYS = ("entailment", "neutral", "contradiction")


def _pearson_pair(xs: list[float], ys: list[float]) -> float:
    n = len(xs)
    if n != len(ys) or n < 2:
        return float("nan")
    mx = sum(xs) / n
    my = sum(ys) / n
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys, strict=True))
    dx = sum((x - mx) ** 2 for x in xs)
    dy = sum((y - my) ** 2 for y in ys)
    den = math.sqrt(dx * dy)
    return float("nan") if den == 0.0 else num / den


def normalized_scores_dict(scores_obj: object) -> dict[str, float]:
    if not isinstance(scores_obj, dict):
        return {}
    out: dict[str, float] = {}
    for key, val in scores_obj.items():
        nk = str(key).strip().lower().replace(" ", "_")
        try:
            out[nk] = float(val)
        except (TypeError, ValueError):
            continue
    return out


by_human_class: defaultdict[str, defaultdict[str, list[float]]] = defaultdict(
    lambda: defaultdict(list)
)
rows_per_human: defaultdict[str, int] = defaultdict(int)
overlap_lex_ent: list[tuple[float, float]] = []

for r in rows:
    m3 = (r.get("baseline_scores") or {}).get("m3") or {}
    nl = m3.get("nli") or {}
    smap = normalized_scores_dict(nl.get("scores") if isinstance(nl, dict) else None)
    if not any(k in smap for k in ENT_KEYS):
        continue

    ha = r.get("human_annotation")
    if not ha:
        continue
    ha_s = str(ha)

    rows_per_human[ha_s] += 1
    for kn in ENT_KEYS:
        if kn in smap:
            by_human_class[ha_s][kn].append(smap[kn])

    ep = smap.get("entailment")
    lr_raw = (m3.get("lexical") or {}).get("sequence_ratio")
    if ep is not None and lr_raw is not None:
        overlap_lex_ent.append((float(lr_raw), ep))

if not by_human_class:
    print(
        "No MNLI-style softmax keys (entailment/neutral/contradiction) in m3.nli.scores — "
        "rerun with --nli or check model labels."
    )
else:
    cols = [k for k in ENT_KEYS if any(by_human_class[h][k] for h in by_human_class)]
    hdr = "{:22}".format("human_annotation")
    hdr += "".join(f"{c[:9]:>10}" for c in cols)
    hdr += "{:>6}".format("rows")
    print("Mean softmax by human_annotation")
    print(hdr)
    print("-" * len(hdr))

    for lbl in sorted(by_human_class):
        pieces = [f"{lbl:22}"]
        for c in cols:
            vals = by_human_class[lbl][c]
            pieces.append(f"{sum(vals) / len(vals):10.4f}" if vals else f"{'—':>10}")
        pieces.append(f"{rows_per_human[lbl]:6d}")
        print("".join(pieces))

    print()
    if len(overlap_lex_ent) >= 2:
        lx, ey = zip(*overlap_lex_ent, strict=True)
        r_le = _pearson_pair(list(lx), list(ey))
        tmpl = "Pearson r (lexical ratio, entailment prob) = {:.4f}  (n={})"
        print(tmpl.format(r_le, len(overlap_lex_ent)))
    else:
        print(
            "Need >=2 rows with lexical + entailment_prob for correlation — skipping r."
        )
